## Looking at results

In [33]:
import pickle
import torch
import numpy as np
from glob import glob
import pandas as pd

In [34]:
folders = [
    'qikprop',
    'atom_edge',
    'edge_qikprop',
    'atom_edge_qikprop'
]

In [35]:
#subfolder_path = f'output/{folders[0]}_3'
subfolder_path = 'output/clintox_atom_edge_qikprop_6'

if True: # Instantiate useful stuffs
    best_aucs = []
    best_auc = None
    best_parameter = None

    data_list = []
    split_list = []
    width_list = []
    depth_list = []
    conv_list = []
    pool_list = []
    max_score_list = []

for path in glob(f'{subfolder_path}/gnn*.out'):
    # print(path)
    valid_aucs = []
    try:
        with open(path, 'r') as f:
            for idx, line in enumerate(f.readlines()):
                if idx < 6: continue
                valid_aucs.append(float(line.split('       ')[1]))
        current_best = max(valid_aucs)
        best_aucs.append(current_best)
        
        #print(current_best, path)
        data_list.append(path.split('/')[1].split('/')[0])
        split_list.append(int(path.split('split=')[1].split('_')[0]))
        width_list.append(int(path.split('width=')[1].split('_')[0]))
        depth_list.append(int(path.split('depth=')[1].split('_')[0]))
        conv_list.append(path.split('conv=')[1].split('_')[0])
        pool_list.append(path.split('pool=')[1].split('_')[0])
        max_score_list.append(current_best)
        
        
        if best_auc is None or current_best > best_auc:
            best_auc = current_best
            best_parameter = path
    except ValueError:
        print(f'Problematic Path: {path}')
        continue
    
df = pd.DataFrame({
    'Dataset': data_list,
    'Split': split_list,
    'Width': width_list,
    'Depth': depth_list,
    'Convolution': conv_list,
    'Pooling': pool_list,
    'Max Val AUROC': max_score_list
})
    
print(df.groupby(['Dataset', 'Width', 'Depth', 'Convolution', 'Pooling']).mean().reset_index().drop(labels='Split', axis=1))

print('')
print('Absolute best AUROC: ', best_auc)            
best_parameter

                       Dataset  Width  Depth Convolution Pooling  \
0  clintox_atom_edge_qikprop_6    256      2         gen    mean   

   Max Val AUROC  
0       0.941838  

Absolute best AUROC:  0.978248


'output/clintox_atom_edge_qikprop_6/gnn-node-py_--data=addAtom_addEdge_qikprop_clintox_--split=0_--width=256_--depth=2_--conv=gen_--pool=mean_--epochs=400.out'